### Note: This Notebook is just for applying model quantization techniques to the best experiment candidate Exp 5 Version 3.0 

In [1]:
import sys
import torch 

""" Run this cell before you run anything. This ensures the modular imports work correctly"""

working_root_directory="/teamspace/studios/this_studio/resnet18_cifar10_classifier"

if working_root_directory not in sys.path: 
    sys.path.append(working_root_directory)

In [2]:
from utils.helper_utils import (
    load_prepare_cifar10,
    build_transforms,
    load_resnet18,
    measure_inference_time,
    get_model_size
    )

from utils.visualization_utils import (
    load_and_plot_metrics_history,
    load_csv_main_model_metrics_history,
    extract_dataframe,
    confusion_matrix_plot,
    each_class_accuracy,
    visualize_per_class_classification_report
    )

from src.test import load_model_for_test,test_model
from src.pruned_model import global_structured_prune
from src.onnx import export_to_onnx
from src.grad_cam import GradCAM,process_input_image
from src.calibrate import calibrate_eval_model,visualize_reliability_diagram
from config.exp_config import dataset_dir,baseline_model_dir,csv_file_path
from torch.utils.data import DataLoader
from typing import List
from copy import deepcopy


In [3]:
#Device
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [4]:
def get_datasets(add_standard_augmentation:bool,add_random_erasing:bool):
        #download cifar10 dataset if it doesn't exist or it is corrupted
        train_dataset,val_dataset,test_dataset=load_prepare_cifar10(
                dataset_path=f"../{dataset_dir}", #notice i have chaged the dataset dir due to current script running jupyter notebook dir
                build_transforms=build_transforms,
                image_size=224,
                standard_aug=add_standard_augmentation,#set False not to be applied standard augmentation or True
                add_random_erasing=add_random_erasing
                )
        return train_dataset,val_dataset,test_dataset 

In [5]:
train_dataset,val_dataset,test_dataset=get_datasets(add_standard_augmentation=False,add_random_erasing=False)

downloaded cifar10 dataset ...


In [6]:
test_loader = DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False,          # no need to shuffle test set
        num_workers=4,
        pin_memory=True,        # speeds up transfer to GPU
        persistent_workers=True # keeps workers alive across epochs
    )

In [19]:
val_loader = DataLoader(
        val_dataset,
        batch_size=128,
        shuffle=False,          # no need to shuffle val set
        num_workers=4,
        pin_memory=True,        # speeds up transfer to GPU
        persistent_workers=True # keeps workers alive across epochs
    )

In [20]:
train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True,          # need to shuffle for train set
        num_workers=4,
        pin_memory=True,        # speeds up transfer to GPU
        persistent_workers=True # keeps workers alive across epochs
    )

In [21]:
exp5_v3_model=load_model_for_test(
    is_baseline=False,
    model_dir=csv_file_path,
    experiment_name="exp5_v3",
    num_classes=10,
    device=device
    )

change the head of resnet18 to 10
--- Extracting Configurations ---
Extracted learning rates from training: [5.8843356115025935e-05, 8.791036923815453e-05, 0.000872719723562627]


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.6, which is newer than your current Lightning version: v2.6.5


In [10]:

history=test_model(
    model=exp5_v3_model,
    test_dataloader=test_loader,
    num_classes=10,
    device=device,
    test_result_dir="./result/exp5_v3"
        )

test model is in progress:   0%|          | 0/79 [00:00<?, ?it/s]

##  Exp 5 Version 3.0 test accuracy before Quantization

In [11]:
print(f"Exp 5 V 3.0 original test accuracy: {history["test_acc"]}")

Exp 5 V 3.0 original test accuracy: 94.31


## Get model size of V 3.0 before Quantized on CPU
#### Note: You can only run inference time and size on CPU for Quantized model

In [12]:
original_v3_mb=get_model_size(exp5_v3_model.model,"../original_v3_inference",experiment_name="exp5_v3")
original_v3_ms=measure_inference_time(model=exp5_v3_model.model,input_shape=(1,3,224,224),save_path="../original_v3_inference",experiment_name="exp5_v3")

In [13]:
print(f"Exp 5 V 3.0 size and inference time before Quantization: {original_v3_mb:.2f} MB | {original_v3_ms:.2f}ms")

Exp 5 V 3.0 size and inference time before Quantization: 42.73 MB | 37.98ms


In [14]:
#Copy model for Quantization (Optional)
static_model=deepcopy(exp5_v3_model.model)
qat_model=deepcopy(exp5_v3_model.model)

## Static Quantization of Exp 5 V 3.0

### Note: Dynamic PTQ quantization technique doesn't help that much with the CNN models like Resnet as it can only quantize the nn.Linear. 

## Static PTQ Quantization to my Exp 5 V 3.0

In [15]:
import torch 
import torch.nn as nn
from tqdm.auto import tqdm
import torch.ao.quantization as tq

class QuantizeResnet18(nn.Module):

    def __init__(self,model:nn.Module): 
        super(QuantizeResnet18,self).__init__() 
        #Fine tuned best resnet model from experiment
        self.model=model
        #QuantStub for converting incoming float32 to Quantized tensors
        self.quantStub=tq.QuantStub()
        #DeQuantStub for converting the Quantized tensors back to float32
        self.dequantStub=tq.DeQuantStub()

    def forward(self,x): 
        #The input pass QuantStub
        x=self.quantStub(x)
        x=self.model(x)
        #DeQuanStub back to original float point
        x=self.dequantStub(x)
        return x  

    def quantize_fuse_modules(self):
        self.eval()
         # Fuse the initial convolution block if present
        if all(hasattr(self.model, name) for name in ("conv1", "bn1", "relu")):
            tq.fuse_modules(
                self.model, ["conv1", "bn1", "relu"], inplace=True
            )

        # Iterate over all residual layers and blocks
        for layer_name in ("layer1", "layer2", "layer3", "layer4"):
            layer = getattr(self.model, layer_name, None)

            if layer is None:
                continue

            for block in layer.children():
                # Resnet18 Block
                if all(hasattr(block, name) for name in ("conv1", "bn1", "relu")):
                    tq.fuse_modules(
                        block, ["conv1", "bn1", "relu"], inplace=True
                    )

                if all(hasattr(block, name) for name in ("conv2", "bn2")):
                    tq.fuse_modules(
                        block, ["conv2", "bn2"], inplace=True
                    )

                # Fuse residual downsample path
                downsample = getattr(block, "downsample", None)

                if isinstance(downsample, nn.Sequential):
                    if (
                        len(downsample) >= 2
                        and isinstance(downsample[0], nn.Conv2d)
                        and isinstance(downsample[1], nn.BatchNorm2d)
                    ):
                        tq.fuse_modules(
                            downsample, ["0", "1"], inplace=True)                   

class StaticPTQ:
    @staticmethod
    def static_ptq_calibrate(model: nn.Module, calibrate_batches: int, device: torch.device, val_dataloader):
        available_batches = len(val_dataloader) 
        requested_calibrate_batches = min(calibrate_batches, available_batches)

        with torch.no_grad(): 
            calibrate_dataLoader = tqdm(val_dataloader, desc="Static PTQ Calibration in progress")

            for idx, (img, _) in enumerate(calibrate_dataLoader): 
                img = img.to(device)
                model(img)
                if idx + 1 >= requested_calibrate_batches: 
                    print(f"Requested calibrate batches ({requested_calibrate_batches}) reached.")
                    break 
            print(f"Calibration finished after processing {requested_calibrate_batches} batches.")
 
    @staticmethod
    def run_resnet_static_ptq(
        model: nn.Module,
        requested_batches: int,
        device: torch.device,
        val_dataLoader
    ): 
        # # fbgemm requires CPU
        # if device.type == 'cuda':
        #     print("Warning: xtbackend requires CPU. Forcing device to CPU for PTQ.")
        #     device = torch.device('cpu')

        quanted_model = QuantizeResnet18(model)
        quanted_model.eval()
        quanted_model.quantize_fuse_modules()
        
        quanted_model.qconfig = tq.get_default_qconfig('x86')
        tq.prepare(quanted_model, inplace=True)      
        quanted_model.to(device)
        
        StaticPTQ.static_ptq_calibrate(
            model=quanted_model,
            calibrate_batches=requested_batches,
            val_dataloader=val_dataLoader,
            device=device
        )

        quanted_model.to("cpu")
        quanted_model = tq.convert(quanted_model, inplace=True)
        return quanted_model


class QATQuantize:
    @staticmethod
    def qat_model_configuration(model: nn.Module):
        qat_model = QuantizeResnet18(model)
        qat_model.eval()
        qat_model.quantize_fuse_modules()
        
        qat_model.qconfig = tq.get_default_qat_qconfig("x86")
        qat_model.train()
        
        qat_model = tq.prepare_qat(qat_model, inplace=False)
        return qat_model 

    @staticmethod
    def train_qat_loop(qat_model: nn.Module, num_epochs: int, device: torch.device, set_lr: float, train_Loader):
        # if device.type == 'cuda':
        #     print("Warning: fbgemm backend requires CPU. Forcing device to CPU for QAT.")
        #     device = torch.device('cpu')

        qat_model.to(device)
        optimizer = torch.optim.SGD(qat_model.parameters(), lr=set_lr, momentum=0.9)
        criterion = torch.nn.CrossEntropyLoss()

        for epoch in range(num_epochs):
            qat_model.train()
            for img, label in tqdm(
                train_Loader,
                desc=f"Epoch {epoch+1}/{num_epochs} | QAT Training"
            ):
                img, label = img.to(device), label.to(device)
                optimizer.zero_grad()
                outputs = qat_model(img)
                loss = criterion(outputs, label)
                loss.backward()
                optimizer.step()
                
        qat_model.eval()
        qat_model.to("cpu")
        qat_model = tq.convert(qat_model, inplace=False)
        return qat_model


In [16]:
print(torch.backends.quantized.supported_engines)

['qnnpack', 'onednn', 'x86', 'fbgemm']


In [41]:
qat_quantized_model=QATQuantize()
qat_train_model=qat_quantized_model.train_qat_loop(qat_model=qat_model,num_epochs=5,device=device,set_lr=1e-4,train_Loader=train_loader)


Epoch 1/5 | QAT Training:   0%|          | 0/313 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [39]:
static_ptq_quantized=StaticPTQ()
static_quanted_model=static_ptq_quantized.run_resnet_static_ptq(model=static_model,requested_batches=50,device=device,val_dataLoader=val_loader)

AssertionError: did not find fuser method for: (<class 'torch.ao.nn.intrinsic.quantized.modules.conv_relu.ConvReLU2d'>, <class 'torch.nn.modules.linear.Identity'>, <class 'torch.nn.modules.linear.Identity'>) 

In [40]:
static_quanted=static_quanted_model.model
print(static_quanted)

ResNet(
  (conv1): QuantizedConvReLU2d(3, 64, kernel_size=(7, 7), stride=(2, 2), scale=0.023669322952628136, zero_point=0, padding=(3, 3))
  (bn1): Identity()
  (relu): Identity()
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.014725316315889359, zero_point=0, padding=(1, 1))
      (bn1): Identity()
      (relu): Identity()
      (conv2): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.03568149358034134, zero_point=71, padding=(1, 1))
      (bn2): Identity()
    )
    (1): BasicBlock(
      (conv1): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.018225062638521194, zero_point=0, padding=(1, 1))
      (bn1): Identity()
      (relu): Identity()
      (conv2): QuantizedConv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.0995955765247345, zero_point=72, padding=(1, 1))
   

In [37]:

for name, module in static_quanted_model.model.named_modules():
    if isinstance(module, (torch.ao.nn.quantized.Linear,
                           torch.ao.nn.quantized.Conv2d)):
        print(f"{name:<30} | {module.weight().dtype}")

conv1                          | torch.qint8
layer1.0.conv1                 | torch.qint8
layer1.0.conv2                 | torch.qint8
layer1.1.conv1                 | torch.qint8
layer1.1.conv2                 | torch.qint8
layer2.0.conv1                 | torch.qint8
layer2.0.conv2                 | torch.qint8
layer2.0.downsample.0          | torch.qint8
layer2.1.conv1                 | torch.qint8
layer2.1.conv2                 | torch.qint8
layer3.0.conv1                 | torch.qint8
layer3.0.conv2                 | torch.qint8
layer3.0.downsample.0          | torch.qint8
layer3.1.conv1                 | torch.qint8
layer3.1.conv2                 | torch.qint8
layer4.0.conv1                 | torch.qint8
layer4.0.conv2                 | torch.qint8
layer4.0.downsample.0          | torch.qint8
layer4.1.conv1                 | torch.qint8
layer4.1.conv2                 | torch.qint8
fc                             | torch.qint8


In [32]:
import os

# Save models
torch.save(exp5_v3_model.state_dict(), "model_w16a16.pth")
torch.save(static_quanted.state_dict(), "model_w8a8.pth")

# Compare file sizes
original_size = os.path.getsize("model_w16a16.pth") / 1024**2
quantized_size = os.path.getsize("model_w8a8.pth") / 1024**2
print(
    f"Size reduction: {original_size / quantized_size:.2f}x ({original_size:.2f}MB -> {quantized_size:.2f}MB)"
)

Size reduction: 3.96x (42.73MB -> 10.78MB)


In [23]:
torch.backends.quantized.engine = 'x86'  # Use 'qnnpack' if on ARM/Mac/Linux Aarch64


In [33]:
static_ptq_v3_mb=get_model_size(static_quanted,"../static_ptq_v3_inference",experiment_name="exp5_v3")
static_ptq_v3_ms=measure_inference_time(model=static_quanted,input_shape=(1,3,224,224),save_path="../static_ptq_v3_inference",experiment_name="exp5_v3")
print(f"Exp 5 V 3.0 size and inference time after static PTQ Quantization: {static_ptq_v3_mb:.2f} MB | {static_ptq_v3_ms:.2f}ms")

NotImplementedError: Could not run 'quantized::conv2d_relu.new' with arguments from the 'CPU' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::conv2d_relu.new' is only available for these backends: [Meta, QuantizedCPU, QuantizedCUDA, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradPrivateUse1, AutogradMeta, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFront

In [36]:

history=test_model(
    model=static_quanted_model.model,
    test_dataloader=val_loader,
    num_classes=10,
    device=device,
    test_result_dir="./result/static_ptq_v3"
        )

test model is in progress:   0%|          | 0/79 [00:00<?, ?it/s]

NotImplementedError: Could not run 'quantized::conv2d_relu.new' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'quantized::conv2d_relu.new' is only available for these backends: [Meta, QuantizedCPU, QuantizedCUDA, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradPrivateUse1, AutogradMeta, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFron

In [25]:
import tempfile
from typing import Tuple 
import time 
def _get_model_size(model,save_path:str,experiment_name:str):

    """
    Create the temporary file path to store model with tempfile
    Get model size in MB

    Args:
    model: Resnet18 Pretrained model
    save_path (str): path for saving model size in dict
    experiment_name (str): Name of experiment


    Return:
    size_mb: convert the model size bytes to MB

    """
    #create save path if not exists
    os.makedirs(save_path,exist_ok=True)
    save_mb_dir=os.path.join(save_path,f"{experiment_name}_MB.pth")

    with tempfile.NamedTemporaryFile(delete=False, suffix=".pth") as tempFile:
        tempfile_path = tempFile.name

    try:
        #Save model dict
        torch.save(model.state_dict(), tempfile_path)

        #Get Resnet18 pretrained model size in bytes
        model_size_bytes = os.path.getsize(tempfile_path)

        #Get model size in Bytes to MB
        size_mb = model_size_bytes / (1024 * 1024)
        #save size MB 
        torch.save({
            "model_size":size_mb
        },
        save_mb_dir
        )

    except Exception as e:
        print(f"Failed measuring the model size due to {e}")

    finally:
        #check if file path still exists
        if os.path.exists(tempfile_path):
            #Remove temporary file path
            os.remove(tempfile_path)
    return size_mb


def _measure_inference_time(
    model:nn.Module,
    input_shape:Tuple[int],
    save_path:str,
    experiment_name:str,
    warm_up_runs=10,
    num_runs=100
    ):
    """
    Measure the inference time of baseline model and  Fine-tuned Resnet50

    Args: 
     model (nn.Module): pretrained resnet18 model 
     input_shape: the image input for model (eg. (1,3,224,224))
     save_path: Save path for model time measurement in dict 
     experiment_name (str): Experiment name 
     warm_up_runs: number of warm-up iterations (default 10)
     num_runs: number of timed runs (default 100)

    Returns:
    avg_time: Average inference timing of the model
    """

    #create save path if not exists
    os.makedirs(save_path,exist_ok=True)
    save_time_dir=os.path.join(save_path,f"{experiment_name}_time.pth")
    #Create sample input data
    sample_input=torch.rand(input_shape).to("cpu")
    #set eval mode
    model.eval()
    model.to("cpu")
    #Run warmup for initializing  system caches and operations
    with torch.no_grad():
        for _ in range(warm_up_runs):
            model(sample_input)

    timings=[]
    with torch.no_grad():
        for _ in range(num_runs):
            #Record start time
            time_start=time.perf_counter()
            #feed sample data to the model
            model(sample_input)
            time_end=time.perf_counter()
            #get the time in seconds and calculate the differences
            timings.append(time_end-time_start)
        #Get the total size and total of the timings
        timings_size=len(timings)
        total_timings=sum(timings)
        #Calculate avg time with average formula and then convert it into miilliseconds
        avg_time=total_timings/timings_size*1000
        #save the time in milliseconds 
        torch.save(
            {
            "model_time":avg_time
            },
            save_time_dir
        )
        return avg_time